# Measles M2 – Modelling and Prediction (Group 3, Northern Region)

**From expected counts to a ranked, uncertainty-aware priority list and a 14-day microplan.**

> **SYNTHETIC TEACHING DATA — NOT FOR PUBLIC HEALTH DECISIONS.** Every count, coverage estimate and outbreak in this pack is simulated (seed 20265024). Results must never be presented as Ghana Health Service findings.

**The Director's decision (fictional, January 2025):** *Which three districts in Northern Region should receive priority measles investigation and immunisation outreach planning, and what evidence supports those choices?* Capacity: **two mobile immunisation outreach teams and one surveillance verification team for 14 days.** Evidence window: January 2023 – December 2024.

| Director's question | Where it is answered |
|---|---|
| Which districts show the strongest combination of case signals and immunisation gaps? | Topics 2.1, 2.3 and 2.5 |
| Do age-specific rates, MCV1/MCV2 coverage and dropout give different priorities from raw counts? | Topics 2.1 and 2.2 (and M1 Topic 1.4) |
| How would incomplete reporting or uncertain vaccination estimates change the recommendation? | Topic 2.5 (and M1 Topic 1.3) |

Input: the hand-off of a Module M1 run (`measles_panel.rds`, `ghana_districts.gpkg`). Outputs: `fig1_burden_map.png`, `fig2_coverage_quality_map.png`, `fig3_uncertainty_alert.png` and `decision_table_northern.csv`.

## Setting up

In [ ]:
# Parameters: the web app (papermill) overrides these for every run.
data_dir <- "/data/measles_model_data"
out_dir  <- "/srv/jobs/outputs"

In [ ]:
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
# the dataset may hold the whole pack (Measles/training_data/...) or just the files: find them
hit <- list.files(data_dir, pattern = "^measles_panel\\.rds$", recursive = TRUE, full.names = TRUE)
if (length(hit) != 1) stop("Expected exactly one measles_panel.rds in the dataset, found ", length(hit), ". Run Module M1, then use its outputs as the dataset.")
data_dir <- dirname(hit[1])
data_dir
list.files(data_dir)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 100)
suppressPackageStartupMessages({
  library(tidyverse); library(sf); library(spdep); library(MASS); library(broom)
  library(mgcv); library(scales); library(patchwork)
})
select <- dplyr::select        # MASS has a select() too; make sure dplyr's wins
options(width = 95)
dir.create(out_dir, showWarnings = FALSE)
focus_region <- "Northern"     # the Director's mandate
set.seed(20250101)             # every simulation in Part 2 is reproducible

---

# Part 2 – Modelling and prediction

**From expected counts to a ranked, uncertainty-aware priority list and a 14-day microplan.**

### Load the data

In [ ]:
panel_file <- file.path(data_dir, "measles_panel.rds")     # the checked panel from Module M1
panel     <- read_rds(panel_file)
districts <- st_read(file.path(data_dir, "ghana_districts.gpkg"), quiet = TRUE)
dim(panel); n_distinct(panel$adm2_code); range(panel$month)
nr_codes <- districts |> st_drop_geometry() |> filter(region == focus_region) |> pull(adm2_code)
nr_map   <- districts |> filter(region == focus_region) |> st_transform(32630)   # UTM 30N: metres, for labelled regional maps
length(nr_codes)
# the same acceptance check as the end of Part 1: never model a panel that fails it
stopifnot(nrow(panel) == 260 * 24, !any(duplicated(panel[, c("adm2_code", "month")])))

---

# Topic 2.1 – Expected cases, age standardisation and the standardised ratio

**Specific objectives**

- Compute expected counts under a common rate, adjusted for population and months reported.
- Repeat the calculation with **age-specific** national rates, because measles burden sits in children and districts differ in age structure.
- Compare the ranking from raw counts, crude rates, age-specific rates and SIR for the 16 Northern districts.
- Quantify how much a ratio can vary by chance with a funnel plot.

**Expected learning outcomes**

By the end of this topic you will be able to:

- Calculate indirectly standardised expected counts, all-age and age-specific.
- Explain to the Director why the ranking by raw counts is not the ranking by risk.
- Read a funnel plot and decide whether a district's excess exceeds chance.

**The annual frame, and who is missing from it**

One row per district for 2024, using only *usable* months (reported at ≥50% and free of impossible values). A district that missed two months is compared over the ten it reported, not penalised with zeros.

In [ ]:
ann <- panel |> filter(year == 2024, usable) |>
  group_by(adm2_code, adm2_name, region, eco_zone) |>
  summarise(O = sum(confirmed), months = n(),
            age_ok = sum(!flag_age_mismatch),                       # months where age parts match the export total
            c_u5 = sum(cases_under5[!flag_age_mismatch]), c_5_14 = sum(cases_5_to_14[!flag_age_mismatch]),
            c_15 = sum(cases_15plus[!flag_age_mismatch]),
            population = first(population), pop_u5 = first(population_under5),
            pop_5_14 = first(population_5_to_14), pop_15 = first(population_15plus),
            mean_reporting = mean(reporting_rate),
            mcv1_prev = first(mcv1_prev), mcv2_prev = first(mcv2_prev), dropout_prev = first(dropout_prev),
            susceptible_prev = first(susceptible_prev),
            mcv1 = first(mcv1_coverage_pct), mcv2 = first(mcv2_coverage_pct), dropout = first(mcv1_to_mcv2_dropout_pct),
            susceptible_pct = first(estimated_susceptible_children_pct), susceptible_n = first(susceptible_children_n),
            children_0_14 = first(children_0_14),
            school = mean(school_contact_index), mobility = mean(estimated_event_mobility_index),
            dens = first(pop_density), fac = first(fac_per_10k), .groups = "drop")
nrow(ann); table(ann$months)
# districts with no usable month in 2024 cannot be modelled: name them, do not drop them silently
districts |> st_drop_geometry() |> filter(!adm2_code %in% ann$adm2_code) |> select(adm2_code, adm2_name, region)
keep <- districts |> filter(adm2_code %in% ann$adm2_code)          # the modelled districts, in map order
nrow(keep)
# months each Northern district contributes: this is already a data-quality statement
ann |> filter(region == focus_region) |> select(adm2_name, months, age_ok, mean_reporting) |>
  mutate(mean_reporting = round(mean_reporting, 1)) |> arrange(months) |> print(n = 16)

**All-age and age-standardised expected cases**

In [ ]:
# all-age: what the district would have had at the national rate, scaled by months reported
rate <- sum(ann$O) / sum(ann$population * ann$months / 12)
round(rate * 1e5, 1)                                                # national confirmed per 100,000 person-years
# age-specific national rates (person-years over the months where the age split is trustworthy)
age_rates <- ann |> summarise(u5   = sum(c_u5)   / sum(pop_u5   * age_ok / 12),
                              a5_14 = sum(c_5_14) / sum(pop_5_14 * age_ok / 12),
                              a15  = sum(c_15)   / sum(pop_15   * age_ok / 12))
round(age_rates * 1e5, 1)                                           # per 100,000 person-years, by age
ann <- ann |> mutate(E = population * months / 12 * rate, SIR = O / E,
                     E_age = months / 12 * (pop_u5 * age_rates$u5 + pop_5_14 * age_rates$a5_14 + pop_15 * age_rates$a15))
ann <- ann |> mutate(E_age = E_age * sum(O) / sum(E_age),           # rescale so the national SIR is exactly 1
                     SIR_age = O / E_age,
                     rate_u5   = c_u5   / (pop_u5   * age_ok / 12) * 1e5,
                     rate_5_14 = c_5_14 / (pop_5_14 * age_ok / 12) * 1e5,
                     rate_all  = O / (population * months / 12) * 1e5)
summary(ann$SIR_age)

**Raw counts against rates: does the ranking change?** (Director's question 2)

In [ ]:
nr_rank <- ann |> filter(region == focus_region) |>
  transmute(adm2_name, cases_2024 = O, months,
            rate_all = round(rate_all, 0), rate_u5 = round(rate_u5, 0), rate_5_14 = round(rate_5_14, 0),
            SIR_age = round(SIR_age, 2),
            rank_cases = rank(-O, ties.method = "min"), rank_u5_rate = rank(-rate_u5, ties.method = "min"),
            rank_SIR_age = rank(-SIR_age, ties.method = "min")) |> arrange(rank_SIR_age)
nr_rank |> print(n = 16)
# how much do the rankings agree? (Spearman correlation between rankings)
nr_rank |> summarise(cases_vs_SIR = round(cor(rank_cases, rank_SIR_age), 2),
                     cases_vs_u5  = round(cor(rank_cases, rank_u5_rate), 2))
# Northern against the other regions, as a comparator only
ann |> group_by(region) |> summarise(SIR_age = round(sum(O) / sum(E_age), 2), rate_u5 = round(sum(c_u5) / sum(pop_u5 * age_ok / 12) * 1e5)) |>
  arrange(-SIR_age) |> head(6)

**How much of that is chance?** Under a Poisson model the SIR of a district at the national rate lies within 1 ± 1.96/√E about 95% of the time.

In [ ]:
ann <- ann |> mutate(lo95 = 1 - 1.96 / sqrt(E_age), hi95 = 1 + 1.96 / sqrt(E_age),
                     outside = SIR_age < lo95 | SIR_age > hi95)
ann |> group_by(northern = region == focus_region) |> summarise(districts = n(), above_95 = sum(SIR_age > hi95), below_95 = sum(SIR_age < lo95))
funnel <- tibble(E = seq(min(ann$E_age), max(ann$E_age), length.out = 200)) |>
  mutate(lo = 1 - 1.96 / sqrt(E), hi = 1 + 1.96 / sqrt(E), lo99 = 1 - 3.09 / sqrt(E), hi99 = 1 + 3.09 / sqrt(E))
ggplot(ann, aes(E_age, SIR_age)) + geom_hline(yintercept = 1, colour = "grey50") +
  geom_line(data = funnel, aes(E, lo), colour = "#C9A063", linewidth = 1) +
  geom_line(data = funnel, aes(E, hi), colour = "#C9A063", linewidth = 1) +
  geom_line(data = funnel, aes(E, hi99), colour = "#C9A063", linetype = 2) +
  geom_point(data = filter(ann, region != focus_region), colour = "grey70", alpha = 0.7) +
  geom_point(data = filter(ann, region == focus_region), colour = "#B2182B", size = 2.5) +
  geom_text(data = filter(ann, region == focus_region), aes(label = adm2_name), size = 2.6, vjust = -0.8, check_overlap = TRUE) +
  scale_x_log10(labels = comma) +
  labs(title = "Funnel plot: age-standardised SIR against expected cases, 2024",
       subtitle = "red: Northern Region districts; grey: comparators. Gold: 95% (solid) and 99.8% (dashed) limits",
       x = "age-standardised expected cases (log scale)", y = "SIR (age-standardised)") + theme_minimal()

**Check yourself**

1. Two Northern districts have similar raw counts but very different under-5 rates. What explains it, and which do you prioritise?
2. Why do we standardise by age for measles, when we did not for malaria?
3. A district reported only 10 of 12 months. Does it get a lower SIR because of that? Why not?

---

# Topic 2.2 – Count regression: are immunity gaps and school contact associated with burden?

**Specific objectives**

- Fit Poisson and negative-binomial regressions for 2024 counts with the age-standardised offset.
- Use the **2023** immunisation estimates as predictors of **2024** burden so the predictor comes before the outcome.
- Handle the strong collinearity between MCV2, dropout and susceptibility honestly.
- Test whether spatial structure remains in the residuals.

**Expected learning outcomes**

- Report a rate ratio for an immunity indicator as an association, not proof.
- Explain why two near-identical predictors cannot both go in one model.
- Use Moran's I on residuals to decide whether a spatial term is needed.

**The immunity indicators move together**

In [ ]:
ann |> select(mcv1_prev, mcv2_prev, dropout_prev, susceptible_prev, school, mobility) |> cor() |> round(2)
# susceptible % is almost a mirror of MCV2 (r close to -1): one of them per model, never both

**Poisson, dispersion, then negative binomial**

In [ ]:
ann <- ann |> mutate(mcv1_gap10 = (95 - mcv1_prev) / 10, dropout10 = dropout_prev / 10, susc10 = susceptible_prev / 10,
                     school10 = school * 10, mob10 = mobility * 10, logdens = log(dens))
fit_p <- glm(O ~ susc10 + dropout10 + school10 + mob10 + logdens + fac + offset(log(E_age)), family = poisson, data = ann)
dispersion <- sum(residuals(fit_p, type = "pearson")^2) / df.residual(fit_p)
round(dispersion, 1)                                     # 1 if Poisson holds
fit_nb <- glm.nb(O ~ susc10 + dropout10 + school10 + mob10 + logdens + fac + offset(log(E_age)), data = ann)
tidy(fit_nb, exponentiate = TRUE, conf.int = TRUE) |> mutate(across(where(is.numeric), ~ round(.x, 3)))
round(fit_nb$theta, 2); AIC(fit_p, fit_nb)

**Which immunity indicator carries the signal?** One indicator at a time, same adjustment set.

In [ ]:
alt <- list(`MCV1 gap (95 - MCV1), per 10 pts` = "mcv1_gap10", `MCV1-MCV2 dropout, per 10 pts` = "dropout10",
            `susceptible children, per 10 pts` = "susc10")
imm_tab <- imap_dfr(alt, function(v, label) {
  f <- as.formula(paste("O ~", v, "+ school10 + mob10 + logdens + fac + offset(log(E_age))"))
  m <- glm.nb(f, data = ann)
  tidy(m, exponentiate = TRUE, conf.int = TRUE) |> filter(term == v) |>
    transmute(indicator = label, RR = estimate, lo = conf.low, hi = conf.high, p = p.value, AIC = AIC(m))
}) |> mutate(across(where(is.numeric), ~ round(.x, 3)))
imm_tab

**Is there anything left in the map?**

In [ ]:
nb_q <- poly2nb(keep, queen = TRUE); names(nb_q) <- keep$adm2_code
lw_q <- nb2listw(nb_q, style = "W", zero.policy = TRUE)
ann  <- ann |> arrange(match(adm2_code, keep$adm2_code))           # same row order as the polygons
table(card(nb_q) == 0)                                             # any district without a neighbour?
ann$res_nb <- residuals(fit_nb, type = "pearson")
moran.test(ann$res_nb, lw_q, zero.policy = TRUE)$estimate[1] |> round(3)
moran.mc(ann$res_nb, lw_q, nsim = 999, zero.policy = TRUE)$p.value

**Figure: rate ratios and the residual map**

In [ ]:
a <- tidy(fit_nb, exponentiate = TRUE, conf.int = TRUE) |> filter(term != "(Intercept)") |>
  mutate(term = recode(term, susc10 = "susceptible % (2023), +10", dropout10 = "dropout % (2023), +10",
                       school10 = "school contact, +0.1", mob10 = "event mobility, +0.1",
                       logdens = "log population density", fac = "facilities per 10k")) |>
  ggplot(aes(estimate, term)) + geom_vline(xintercept = 1, colour = "grey50") +
  geom_errorbar(aes(xmin = conf.low, xmax = conf.high), width = 0.2, colour = "#010066") +
  geom_point(colour = "#010066", size = 2.5) + scale_x_log10() +
  labs(title = "Negative-binomial model: rate ratios (2024 cases)",
       subtitle = "offset: age-standardised expected cases. Associations, not proof.",
       x = "rate ratio (log scale)", y = NULL) + theme_minimal()
b <- districts |> left_join(ann |> select(adm2_code, res_nb), by = "adm2_code") |> ggplot() +
  geom_sf(aes(fill = res_nb), colour = "white", linewidth = 0.1) +
  geom_sf(data = st_union(districts |> filter(region == focus_region)), fill = NA, colour = "black", linewidth = 0.4) +
  scale_fill_gradient2("Pearson\nresidual", low = "#2166AC", mid = "white", high = "#B2182B",
                       midpoint = 0, limits = c(-3, 3), oob = squish) +
  labs(title = "What the covariates missed", subtitle = "black outline: Northern Region") + theme_void()
a + b + plot_layout(widths = c(1.2, 1))

**Check yourself**

1. The susceptible-children rate ratio is above 1. Write the sentence you would say to the Director, without the word "cause".
2. Why were the 2023 coverage estimates used, not the 2024 ones?
3. If Moran's I on the residuals is clearly positive, what does that tell you to do next?

---

# Topic 2.3 – Borrowing strength from neighbours: smoothed risk with uncertainty

**Specific objectives**

- Fit a hierarchical spatial model (intrinsic CAR / Markov random field) for 2024 age-standardised risk.
- Obtain posterior relative risks with credible intervals and **exceedance probabilities** by simulation.
- Rank the Northern districts with a probability attached to each rank.

**Expected learning outcomes**

- Explain why a small district's estimate moves toward its neighbours and a large one's does not.
- Report "P(RR > 1)" and "P(in the regional top 3)" instead of a bare rank.

The spatial-only model is used here deliberately: it describes the **observed case signal**. The immunity gap enters the decision separately in Topic 2.5, so it is not counted twice.

In [ ]:
ann$id <- factor(ann$adm2_code, levels = keep$adm2_code)
m_sp <- gam(O ~ s(id, bs = "mrf", xt = list(nb = nb_q)) + offset(log(E_age)),
            family = nb(), data = ann, method = "REML")
summary(m_sp)$s.table
# posterior relative risk: 1,000 draws of the coefficients pushed through the model
Xp   <- predict(m_sp, type = "lpmatrix")
beta <- rmvn(1000, coef(m_sp), vcov(m_sp))
RR   <- exp(Xp %*% t(beta))                                        # districts x 1,000 draws
ann  <- ann |> mutate(RR_mean = rowMeans(RR),
                      RR_lo = apply(RR, 1, quantile, 0.025), RR_hi = apply(RR, 1, quantile, 0.975),
                      P_RR_gt1 = rowMeans(RR > 1))

**The Northern ranking, with probabilities**

In [ ]:
nr_idx  <- which(ann$region == focus_region)
rank_nr <- apply(RR[nr_idx, ], 2, function(x) rank(-x))            # rank of each Northern district in every draw
ann$P_top3_RR <- NA_real_
ann$P_top3_RR[nr_idx] <- rowMeans(rank_nr <= 3)
ann |> filter(region == focus_region) |>
  select(adm2_name, O, E_age, SIR_age, RR_mean, RR_lo, RR_hi, P_RR_gt1, P_top3_RR) |>
  mutate(across(where(is.numeric), ~ round(.x, 2))) |> arrange(-RR_mean) |> print(n = 16)
# shrinkage, measured: which districts moved most?
ann |> summarise(sd_SIR = round(sd(SIR_age), 3), sd_RR = round(sd(RR_mean), 3), cor = round(cor(SIR_age, RR_mean), 3))

**Figure 1 – Observed risk and burden map (Northern Region)**

Left: posterior age-standardised relative risk (smoothed). Right: under-5 confirmed rate. Districts are labelled so the map can be read at the review meeting without the table.

In [ ]:
nr_fig <- nr_map |> left_join(ann |> select(adm2_code, RR_mean, P_RR_gt1, rate_u5, O), by = "adm2_code")
a <- ggplot(nr_fig) + geom_sf(aes(fill = RR_mean), colour = "white", linewidth = 0.3) +
  geom_sf_text(aes(label = paste0(adm2_name, "\n", round(RR_mean, 2))), size = 2.2) +
  scale_fill_gradient2("posterior RR", low = "#2166AC", mid = "white", high = "#B2182B", midpoint = 1) +
  labs(title = "Age-standardised relative risk, 2024", subtitle = "1 = national level; spatially smoothed") + theme_void()
b <- ggplot(nr_fig) + geom_sf(aes(fill = rate_u5), colour = "white", linewidth = 0.3) +
  geom_sf_text(aes(label = paste0(adm2_name, "\n", round(rate_u5))), size = 2.2) +
  scale_fill_distiller("per 100k\nperson-years", palette = "Reds", direction = 1) +
  labs(title = "Under-5 confirmed measles rate, 2024", subtitle = "usable months only") + theme_void()
fig1 <- a + b + plot_annotation(title = "Figure 1. Measles burden, Northern Region (synthetic data)")
ggsave(file.path(out_dir, "fig1_burden_map.png"), fig1, width = 12, height = 6.5, dpi = 200, bg = "white")
fig1

**Check yourself**

1. A district's SIR is 1.9 and its posterior RR is 1.3 with interval 0.9–1.8. What do you tell its District Director?
2. Why is "P(in the Northern top 3)" more honest than a rank for the decision note?

---

# Topic 2.4 – Predicting next month, and turning the forecast into an alert

**Specific objectives**

- Write the prediction statement and test every predictor for leakage.
- Judge every model against naive baselines on months it never saw (Jul–Dec 2024).
- Produce prediction intervals and check their coverage.
- Convert the predictive distribution into an **alert probability** and evaluate alerts on the holdout.
- Produce a January 2025 forecast for the 16 Northern districts.

In [ ]:
tibble(item = c("Outcome", "Unit", "Prediction made", "Horizon", "Predictors allowed",
                "Training period", "Test period", "Metric", "Use"),
       value = c("confirmed measles per 100,000 (monthly)", "district-month",
                 "end of month t, once the DHIMS2 export for t is in", "month t + 1",
                 "anything known at the end of month t", "Jan 2023 - Jun 2024 (18 months)",
                 "Jul - Dec 2024 (6 months)", "MAE, RMSE, 90% interval coverage, alert sensitivity/PPV",
                 "flag districts for verification before the outreach cycle")) |> print(n = 10)
tibble(column = c("cases_lag1", "school_contact_index (same month)", "estimated_event_mobility_index (same month)",
                  "mobility_lag1", "susceptible_prev, dropout_prev (2023 estimates)", "month_of_year", "confirmed"),
       known_when_predicting_t_plus_1 = c("yes: last month's export", "yes: the term calendar is published in advance",
                                          "NO: estimated during the month", "yes", "yes: published annual estimates",
                                          "yes: the calendar", "NO: it is the outcome")) |> print(n = 10)

**The feature set and the split**

In [ ]:
model_df <- panel |> arrange(adm2_code, month) |>
  mutate(inc_100k = confirmed / population * 1e5,
         inc_lag1 = cases_lag1 / population * 1e5,                 # cases_lag1 is NA when last month was not usable
         susc10 = susceptible_prev / 10, dropout10 = dropout_prev / 10, school10 = school_contact_index * 10,
         mob_lag10 = mobility_lag1 * 10, logdens = log(pop_density),
         id = factor(adm2_code, levels = keep$adm2_code)) |>
  filter(usable, !is.na(inc_lag1), !is.na(mobility_lag1), adm2_code %in% keep$adm2_code)
model_df |> count(split)
train <- model_df |> filter(split == "train"); test <- model_df |> filter(split == "test")
range(test$month)
# the alert threshold: a district-month in the top 10% of training-period incidence nationally
alert_thr <- quantile(train$inc_100k, 0.9)
round(alert_thr, 1)                                                # confirmed per 100,000 in one month
test <- test |> mutate(alert_obs = inc_100k >= alert_thr)

**The baselines**

In [ ]:
rmse <- function(obs, pred) sqrt(mean((obs - pred)^2)); mae <- function(obs, pred) mean(abs(obs - pred))
seasonal <- train |> group_by(adm2_code, month_of_year) |> summarise(inc_seasonal = mean(inc_100k), .groups = "drop")
test <- test |> left_join(seasonal, by = c("adm2_code", "month_of_year")) |>
  mutate(inc_seasonal = if_else(is.na(inc_seasonal), inc_lag1, inc_seasonal))
score <- function(name, pred) tibble(model = name, MAE = round(mae(test$inc_100k, pred), 2), RMSE = round(rmse(test$inc_100k, pred), 2))
scores <- bind_rows(score("Naive: last month", test$inc_lag1), score("Naive: same month last year", test$inc_seasonal))
scores

**The regression forecast, then the spatio-temporal model**

In [ ]:
f_glm <- confirmed ~ log1p(inc_lag1) + school10 + mob_lag10 + susc10 + dropout10 + factor(month_of_year) +
  logdens + fac_per_10k + offset(log(population))
fit_fc <- glm.nb(f_glm, data = train)
tidy(fit_fc, exponentiate = TRUE, conf.int = TRUE) |> filter(!str_detect(term, "month_of_year")) |>
  mutate(across(where(is.numeric), ~ round(.x, 3)))
test$pred_glm <- as.numeric(predict(fit_fc, newdata = test, type = "response")) / test$population * 1e5
f_st <- confirmed ~ log1p(inc_lag1) + school10 + mob_lag10 + susc10 + dropout10 + logdens + fac_per_10k +
  s(month_of_year, bs = "cc", k = 8) + s(id, bs = "mrf", xt = list(nb = nb_q)) + offset(log(population))
m_st <- gam(f_st, family = nb(), data = train, method = "REML", knots = list(month_of_year = c(0.5, 12.5)))
summary(m_st)$s.table
test$pred_st <- as.numeric(predict(m_st, newdata = test, type = "response")) / test$population * 1e5
scores <- bind_rows(scores, score("NB regression", test$pred_glm), score("Spatio-temporal (mgcv)", test$pred_st)) |>
  mutate(skill_MAE_vs_last_month = round(1 - MAE / MAE[1], 2))
scores
# the same scores for Northern district-months only: the region the decision is about
test |> filter(region == focus_region) |>
  summarise(n = n(), MAE_last_month = round(mae(inc_100k, inc_lag1), 2), MAE_seasonal = round(mae(inc_100k, inc_seasonal), 2),
            MAE_glm = round(mae(inc_100k, pred_glm), 2), MAE_st = round(mae(inc_100k, pred_st), 2))

**Prediction intervals, coverage and alert performance**

In [ ]:
sim_pred <- function(m, newdata, n = 500) {                      # draws of next month's count per row
  Xp   <- predict(m, newdata = newdata, type = "lpmatrix")
  beta <- rmvn(n, coef(m), vcov(m))
  mu   <- exp(Xp %*% t(beta) + log(newdata$population))
  matrix(rnbinom(length(mu), size = m$family$getTheta(TRUE), mu = mu), nrow = nrow(mu))
}
draws <- sim_pred(m_st, test) / test$population * 1e5
test  <- test |> mutate(st_lo = apply(draws, 1, quantile, 0.05), st_hi = apply(draws, 1, quantile, 0.95),
                        covered = inc_100k >= st_lo & inc_100k <= st_hi,
                        p_alert = rowMeans(draws >= alert_thr))
test |> group_by(scope = if_else(region == focus_region, "Northern", "Other regions")) |>
  summarise(coverage_90 = round(mean(covered), 2), median_width = round(median(st_hi - st_lo), 1))
# at what probability should we raise an alert? A cut of 0.5 is rarely right for rare, noisy events.
# Choose the cut on the TRAINING months (best sensitivity + specificity), then judge it on the holdout.
auc <- function(p, y) { r <- rank(p); n1 <- sum(y); n0 <- sum(!y); (sum(r[y]) - n1 * (n1 + 1) / 2) / (n1 * n0) }
p_train <- rowMeans(sim_pred(m_st, train, 300) / train$population * 1e5 >= alert_thr)
y_train <- train$inc_100k >= alert_thr
cuts <- seq(0.02, 0.5, by = 0.01)
alert_cut <- cuts[which.max(sapply(cuts, function(k) mean(p_train[y_train] >= k) + mean(p_train[!y_train] < k)))]
alert_cut
perf <- function(p, y, k) tibble(high_months = sum(y), alerts_raised = sum(p >= k),
                                 sensitivity = round(mean(p[y] >= k), 2), specificity = round(mean(p[!y] < k), 2),
                                 PPV = round(sum(p >= k & y) / max(1, sum(p >= k)), 2), AUC = round(auc(p, y), 2))
bind_rows(train = perf(p_train, y_train, alert_cut), test = perf(test$p_alert, test$alert_obs, alert_cut), .id = "period")
# can the baselines rank high months as well? (AUC 0.5 = no better than chance)
tibble(ranker = c("model P(alert)", "last month", "same month last year"),
       AUC_test = round(c(auc(test$p_alert, test$alert_obs), auc(test$inc_lag1, test$alert_obs),
                          auc(test$inc_seasonal, test$alert_obs)), 2))
# when did the holdout's high months happen? An off-season wave the training seasonality never saw is hard to forecast.
test |> filter(alert_obs) |> count(month)

**Forecast for January 2025 (Northern Region)**

The model is refitted on all 24 months. Inputs for January 2025: December 2024 cases (or the last usable month if December was not usable — stated in the table), December 2024 mobility, the January school-contact value from the term calendar (January 2024 used as the published-calendar proxy, an assumption to confirm with the Education Service), and the 2023 immunisation estimates.

In [ ]:
m_all <- gam(f_st, family = nb(), data = model_df, method = "REML", knots = list(month_of_year = c(0.5, 12.5)))
last_usable <- panel |> filter(usable, region == focus_region) |> group_by(adm2_code) |> slice_max(month, n = 1) |>
  ungroup() |> transmute(adm2_code, lag_month = month, inc_lag1 = confirmed / population * 1e5)
jan <- panel |> filter(region == focus_region, month == ymd("2024-12-01")) |>
  transmute(adm2_code, adm2_name, region, population, pop_density, fac_per_10k, susceptible_prev, dropout_prev,
            mob_lag10 = estimated_event_mobility_index * 10) |>
  left_join(last_usable, by = "adm2_code") |>
  left_join(panel |> filter(month == ymd("2024-01-01")) |> transmute(adm2_code, school10 = school_contact_index * 10), by = "adm2_code") |>
  mutate(month_of_year = 1, susc10 = susceptible_prev / 10, dropout10 = dropout_prev / 10, logdens = log(pop_density),
         id = factor(adm2_code, levels = keep$adm2_code))
jd  <- sim_pred(m_all, jan, n = 2000)
jan <- jan |> mutate(fc_cases = rowMeans(jd), fc_lo = apply(jd, 1, quantile, 0.05), fc_hi = apply(jd, 1, quantile, 0.95),
                     p_alert = rowMeans(jd / population * 1e5 >= alert_thr))
jan |> select(adm2_name, lag_month, fc_cases, fc_lo, fc_hi, p_alert) |>
  mutate(across(c(fc_cases, p_alert), ~ round(.x, 2))) |> arrange(-p_alert) |> print(n = 16)

**Figure: holdout forecasts with intervals, against what happened** (four Northern districts)

In [ ]:
show <- jan |> slice_max(p_alert, n = 4) |> pull(adm2_name)
test |> filter(adm2_name %in% show) |>
  ggplot(aes(month)) +
  geom_ribbon(aes(ymin = st_lo, ymax = st_hi), fill = "#010066", alpha = 0.15) +
  geom_line(aes(y = pred_st), colour = "#010066", linewidth = 1) +
  geom_point(aes(y = inc_100k), colour = "#B2182B", size = 2) +
  geom_hline(yintercept = alert_thr, colour = "#C9A063", linetype = 2) +
  facet_wrap(~ adm2_name, scales = "free_y") +
  labs(title = "Holdout months: forecast with a 90% prediction interval, against what happened",
       subtitle = "blue: model; red: observed; gold dashed: alert threshold", x = NULL, y = "confirmed per 100,000 (month)") +
  theme_minimal()

**Check yourself**

1. The model beats "last month" by only a small margin on MAE. Give two reasons to use it anyway, and one reason not to.
2. The 90% intervals cover less than 90% of the Northern months. What does that mean for how you word the alert?
3. Why must the January school value come from the published calendar and not from the January export?

---

# Topic 2.5 – The decision: a validation-aware priority list and a 14-day microplan

**Specific objectives**

- Combine four evidence domains for each Northern district: **case signal** (posterior RR), **immunity gap** (estimated susceptible children), **programme failure** (MCV1→MCV2 dropout), and **near-term risk** (January alert probability).
- Propagate uncertainty in each domain by simulation and report **P(in the top 3)**.
- Test the ranking against incomplete reporting and uncertain vaccination estimates (Director's question 3).
- Allocate two outreach teams and one verification team for 14 days.

The domains are ranked within Northern Region (1 = worst) and averaged with weights. **The weights are validation-aware:** the forecast earned only modest skill and weak alert sensitivity on the holdout (Topic 2.4), so it gets 10%; the three descriptive domains get 30% each. The weights are a judgement, written in the code so the Directorate can challenge them.

In [ ]:
nr_dec <- ann |> filter(region == focus_region) |>
  select(adm2_code, adm2_name, O, months, mean_reporting, RR_mean, RR_lo, RR_hi, P_RR_gt1, rate_u5,
         mcv1, mcv2, dropout, susceptible_pct, susceptible_n, children_0_14) |>
  left_join(jan |> select(adm2_code, p_alert, fc_cases), by = "adm2_code")
# the data-quality picture for the 16 districts, over the whole 24 months
dq <- panel |> filter(region == focus_region) |> group_by(adm2_code) |>
  summarise(months_usable_24 = sum(usable), months_missing = sum(is.na(confirmed)),
            months_lt80 = sum(!is.na(reporting_rate) & reporting_rate < 80),
            flags_impossible = sum(flag_conf_gt_test | flag_test_gt_susp), .groups = "drop")
nr_dec <- nr_dec |> left_join(dq, by = "adm2_code")
w <- c(case = 0.30, susceptible = 0.30, dropout = 0.30, forecast = 0.10)       # the weights, in plain sight
score_fun <- function(case, susc, drop, fc) {
  w["case"] * rank(-case) + w["susceptible"] * rank(-susc) + w["dropout"] * rank(-drop) + w["forecast"] * rank(-fc)
}
nr_dec <- nr_dec |> mutate(score = score_fun(RR_mean, susceptible_n, dropout, p_alert), priority = rank(score, ties.method = "first"))

**Uncertainty, propagated.** In each of 2,000 draws: the case signal is a posterior RR draw; the susceptible % and dropout are perturbed by ±(sd) percentage points because administrative coverage rests on uncertain denominators; the alert probability is resampled from its forecast. Change `cov_sd` to see how fragile the ranking is.

In [ ]:
cov_sd <- 4                                                    # assumed sd (percentage points) of the coverage-based estimates
n_sim  <- 2000
RR_nr  <- RR[match(nr_dec$adm2_code, ann$adm2_code), sample(ncol(RR), n_sim, replace = TRUE)]
# the alert probability is itself an estimate from 2,000 forecast draws: resample it as a binomial proportion
top3  <- matrix(FALSE, nrow(nr_dec), n_sim)
for (s in seq_len(n_sim)) {
  susc_s <- pmin(pmax(nr_dec$susceptible_pct + rnorm(nrow(nr_dec), 0, cov_sd), 0), 100) / 100 * nr_dec$children_0_14
  drop_s <- pmax(nr_dec$dropout + rnorm(nrow(nr_dec), 0, cov_sd), 0)
  fc_s   <- rbeta(nrow(nr_dec), 1 + nr_dec$p_alert * 200, 1 + (1 - nr_dec$p_alert) * 200)
  top3[, s] <- rank(score_fun(RR_nr[, s], susc_s, drop_s, fc_s), ties.method = "random") <= 3
}
nr_dec$P_top3 <- rowMeans(top3)
# the recommendation is ordered by P(top 3), the rank that survives the uncertainty; the point-estimate rank is kept beside it
nr_dec <- nr_dec |> mutate(rank_point = priority, priority = rank(-P_top3 + score / 1e4, ties.method = "first"))
nr_dec |> arrange(priority) |> select(adm2_name, rank_point, P_top3, priority) |> head(6)

**Sensitivity: would the answer change?**

| Scenario | What it tests |
|---|---|
| Main | 50% reporting cut, weights 30/30/30/10 |
| Reporting ≥80% | stricter definition of a usable month (Part 1 sensitivity flag) |
| Completeness-adjusted counts | inflates each month's count by 100 / reporting rate: under-reporting assumed proportional |
| Equal weights | the forecast counted as much as the other domains |
| Cases only | what a ranking by case signal alone would recommend |
| Immunity only | what a ranking by susceptibles + dropout alone would recommend |
| Raw counts | the naive answer the Director warned against |

In [ ]:
refit_rr <- function(dat) {                         # re-run Topic 2.1 + 2.3 on an alternative case table
  a <- dat |> group_by(adm2_code) |> summarise(O = round(sum(cases)), months = n(), .groups = "drop") |>
    right_join(ann |> select(adm2_code, region, E_age, months0 = months), by = "adm2_code") |>
    mutate(O = replace_na(O, 0L), months = replace_na(months, 0L), E = E_age / months0 * months)
  a <- a |> filter(months > 0) |> mutate(E = E * sum(O) / sum(E), id = factor(adm2_code, levels = keep$adm2_code))
  m <- gam(O ~ s(id, bs = "mrf", xt = list(nb = nb_q)) + offset(log(E)), family = nb(), data = a, method = "REML",
           drop.unused.levels = FALSE)              # keep every district in the neighbour structure
  a |> mutate(RR = as.numeric(exp(predict(m) - log(E)))) |> filter(region == focus_region) |> select(adm2_code, RR)
}
rr80  <- refit_rr(panel |> filter(year == 2024, usable, reported_80) |> transmute(adm2_code, cases = confirmed))
rradj <- refit_rr(panel |> filter(year == 2024, usable) |> transmute(adm2_code, cases = confirmed * 100 / pmax(reporting_rate, 50)))
sens <- nr_dec |> select(adm2_code, adm2_name, RR_mean, susceptible_n, dropout, p_alert, O, rank_point) |>
  left_join(rr80 |> rename(RR80 = RR), by = "adm2_code") |> left_join(rradj |> rename(RRadj = RR), by = "adm2_code") |>
  mutate(rank_main     = rank_point,
         rank_rep80    = rank(score_fun(RR80, susceptible_n, dropout, p_alert), ties.method = "first"),
         rank_complete = rank(score_fun(RRadj, susceptible_n, dropout, p_alert), ties.method = "first"),
         rank_equal_w  = rank(rank(-RR_mean) + rank(-susceptible_n) + rank(-dropout) + rank(-p_alert), ties.method = "first"),
         rank_cases    = rank(-RR_mean, ties.method = "first"),
         rank_immunity = rank(rank(-susceptible_n) + rank(-dropout), ties.method = "first"),
         rank_raw      = rank(-O, ties.method = "first"))
sens |> select(adm2_name, starts_with("rank_")) |> arrange(rank_main) |> print(n = 16)
# which districts are top 3 in each scenario?
sens |> summarise(across(starts_with("rank_"), ~ paste(adm2_name[order(.x)][1:3], collapse = ", "))) |>
  pivot_longer(everything(), names_to = "scenario", values_to = "top_3")

**The ranked decision table**

In [ ]:
decision <- nr_dec |> left_join(sens |> select(adm2_code, rank_rep80, rank_complete), by = "adm2_code") |>
  arrange(priority) |>
  transmute(priority, district = adm2_name,
            `P(top 3)` = round(P_top3, 2), `rank (point estimate)` = rank_point,
            `cases 2024` = O, `RR (95% CrI)` = sprintf("%.2f (%.2f-%.2f)", RR_mean, RR_lo, RR_hi),
            `<5 rate /100k` = round(rate_u5),
            `MCV1 %` = mcv1, `MCV2 %` = mcv2, `dropout %` = dropout,
            `susceptible children` = round(susceptible_n),
            `P(alert) Jan 2025` = round(p_alert, 2),
            `usable months /24` = months_usable_24, `months <80% reporting` = months_lt80,
            `rank if >=80% reporting` = rank_rep80, `rank if completeness-adjusted` = rank_complete,
            evidence_quality = case_when(months_usable_24 >= 23 & months_lt80 <= 4 ~ "good",
                                         months_usable_24 >= 21 ~ "fair: some months missing",
                                         TRUE ~ "weak: verify before acting"))
decision |> print(n = 16, width = 250)
write_csv(decision, file.path(out_dir, "decision_table_northern.csv"))

**Figure 2 – Coverage gaps and surveillance quality (Northern Region)**

In [ ]:
nr_q <- nr_map |> left_join(nr_dec |> select(adm2_code, mcv1, dropout, susceptible_n, months_usable_24, mean_reporting), by = "adm2_code")
qmap <- function(var, title, pal, dir = 1, lab = waiver()) {
  ggplot(nr_q) + geom_sf(aes(fill = .data[[var]]), colour = "white", linewidth = 0.3) +
    geom_sf_text(aes(label = adm2_name), size = 1.9) +
    scale_fill_distiller(NULL, palette = pal, direction = dir, labels = lab) + labs(title = title) + theme_void()
}
fig2 <- (qmap("mcv1", "MCV1 coverage 2024 (%)", "RdYlGn", 1) + qmap("dropout", "MCV1-MCV2 dropout 2024 (%)", "Oranges")) /
  (qmap("susceptible_n", "Estimated susceptible children 0-14", "Purples", 1, comma) +
     qmap("months_usable_24", "Usable reporting months (of 24)", "Greys", -1)) +
  plot_annotation(title = "Figure 2. Immunity gaps and surveillance quality, Northern Region (synthetic data)",
                  subtitle = "coverage estimates are administrative and uncertain; pale districts in the last panel have the weakest evidence")
ggsave(file.path(out_dir, "fig2_coverage_quality_map.png"), fig2, width = 11, height = 10, dpi = 200, bg = "white")
fig2

**Figure 3 – Model uncertainty and alert**

Left: probability that each district is in the top 3 after propagating case, coverage and forecast uncertainty. Right: January 2025 forecast with its 90% interval and the alert probability.

In [ ]:
a <- nr_dec |> ggplot(aes(P_top3, reorder(adm2_name, P_top3))) +
  geom_col(aes(fill = priority <= 3), width = 0.7) +
  scale_fill_manual(values = c(`TRUE` = "#B2182B", `FALSE` = "grey70"), labels = c("other", "recommended top 3"), name = NULL) +
  scale_x_continuous(labels = percent, limits = c(0, 1)) +
  labs(title = "P(in the Northern top 3)", subtitle = paste0(n_sim, " simulations; coverage sd = ", cov_sd, " points"),
       x = NULL, y = NULL) + theme_minimal() + theme(legend.position = "bottom")
b <- jan |> ggplot(aes(y = reorder(adm2_name, p_alert))) +
  geom_errorbar(aes(xmin = fc_lo, xmax = fc_hi), width = 0.3, colour = "#010066") +
  geom_point(aes(x = fc_cases, colour = p_alert), size = 3) +
  scale_colour_gradient("P(alert)", low = "#FDD49E", high = "#B2182B", limits = c(0, max(jan$p_alert))) +
  labs(title = "January 2025 forecast: confirmed cases", subtitle = "point: mean; bar: 90% prediction interval",
       x = "confirmed cases (month)", y = NULL) + theme_minimal()
fig3 <- a + b + plot_annotation(title = "Figure 3. How certain is the ranking, and what is expected next month? (synthetic data)")
ggsave(file.path(out_dir, "fig3_uncertainty_alert.png"), fig3, width = 12, height = 6, dpi = 200, bg = "white")
fig3

**The 14-day microplan**

Two outreach teams (28 team-days) go to the top 3 in proportion to the estimated susceptible children; the verification team visits each of the three for case verification and then spends its remaining days on the **runner-up district whose place in the ranking is most likely to be an artefact of weak reporting** (highest P(top 3) × months missing or below 80%). That visit is how the contested third slot gets settled. The daily capacity is an **exercise assumption** to be replaced with the programme's own figure.

In [ ]:
team_capacity_per_day <- 250                                   # children reached per outreach team per day (assumption)
top <- decision |> slice(1:3)
plan <- nr_dec |> filter(adm2_name %in% top$district) |> arrange(priority) |>
  mutate(share = susceptible_n / sum(susceptible_n),
         outreach_team_days = round(share * 2 * 14),           # 2 teams x 14 days = 28 team-days
         children_reachable = outreach_team_days * team_capacity_per_day,
         pct_of_susceptibles = round(children_reachable / susceptible_n * 100))
plan$outreach_team_days[1] <- plan$outreach_team_days[1] + (28 - sum(plan$outreach_team_days))   # keep the total at 28
verify_extra <- nr_dec |> filter(!adm2_name %in% top$district) |>
  mutate(need = P_top3 * (months_missing + months_lt80)) |> slice_max(need, n = 1)   # a contender whose evidence is weakest
plan |> transmute(priority, district = adm2_name, susceptible_children = round(susceptible_n), outreach_team_days,
                  children_reachable, pct_of_susceptibles,
                  verification = paste0("days ", c("1-3", "4-6", "7-9"), ": line list, lab results, active case search"))
# the schedule follows the team-days: team A starts in district 1, team B in district 2, both finish in district 3
d <- plan$outreach_team_days
stopifnot(d[1] <= 14, d[2] <= 14, sum(d) == 28)
tibble(team = c("Outreach team A", "Outreach team A", "Outreach team B", "Outreach team B", "Verification team", "Verification team"),
       days = c(paste0("1-", d[1]), paste0(d[1] + 1, "-14"), paste0("1-", d[2]), paste0(d[2] + 1, "-14"), "1-9", "10-14"),
       district = c(plan$adm2_name[1], plan$adm2_name[3], plan$adm2_name[2], plan$adm2_name[3],
                    paste(plan$adm2_name, collapse = " -> "), verify_extra$adm2_name),
       task = c("MCV1/MCV2 catch-up, defaulter tracing", "catch-up", "MCV1/MCV2 catch-up, defaulter tracing", "catch-up",
                "case verification, 3 days per district", "data verification: missing and low-completeness months"))

**What could change our mind** (for the decision note)

- **Verification finds the excess is a reporting artefact** (e.g. a backlog entered in one month, or non-measles rash illness): the case signal for that district falls and the next district by P(top 3) moves up.
- **A coverage survey contradicts administrative MCV1/MCV2**: the immunity domain is the most uncertain input; re-run Topic 2.5 with survey values.
- **Districts with missing months** may be under-ranked: compare `rank if >=80% reporting` and `rank if completeness-adjusted` in the decision table before the review.
- **January cases above the forecast interval** in a district outside the top 3: that is an alert to trigger investigation, regardless of rank.

**Check yourself**

1. A district has the highest P(alert) but rank 6 overall. Do you send the verification team there? Why?
2. Increase `cov_sd` to 8. Which district's P(top 3) changes most, and what does that tell you about the evidence behind it?
3. Write the one-sentence uncertainty statement that must accompany the ranking in the decision note.

---

## End of the notebook

Outputs written to `outputs/`: `fig1_burden_map.png`, `fig2_coverage_quality_map.png`, `fig3_uncertainty_alert.png` and `decision_table_northern.csv` — the three figures and the ranked table the commission asks for.

Save a copy of this notebook to your own Drive before you close it (File → Save a copy in Drive). The conversation is not a record; the script is.

> All results are from **synthetic** teaching data and must not be presented as Ghana Health Service findings.

## Export for the web map viewer

Writes the map layers listed in `layers.json`.

In [ ]:
# ---- Export for the web map viewer (added by tools/prepare_notebooks.py) ----
# Each block below is wrapped in safe(): if a step was edited or skipped, the other layers are
# still written. The viewer discovers everything through layers.json.
layers <- list()
safe <- function(label, expr) tryCatch(expr, error = function(e)
  message("Export skipped - ", label, ": ", conditionMessage(e)))
add_layer <- function(...) layers[[length(layers) + 1]] <<- list(...)
wgs <- function(x) sf::st_transform(x, 4326)

# district polygons, in lon/lat, lightly simplified for the web
web_districts <- function(d, tol = 0.002) {
  d <- wgs(d)
  keep <- intersect(c("adm2_code", "adm2_name", "region", "eco_zone"), names(d))
  suppressWarnings(sf::st_simplify(d[, keep], preserveTopology = TRUE, dTolerance = tol))
}
write_geojson <- function(x, name) {
  f <- file.path(out_dir, paste0(name, ".geojson"))
  if (file.exists(f)) file.remove(f)
  x <- dplyr::mutate(x, dplyr::across(dplyr::where(is.numeric), ~ round(.x, 4)))
  sf::st_write(x, f, driver = "GeoJSON", quiet = TRUE, layer_options = "COORDINATE_PRECISION=5")
}
# long district-month table -> one column per month (inc_2024_01, ...) + field list for the slider
widen_months <- function(df, value, prefix) {
  df <- as.data.frame(df)
  d  <- data.frame(adm2_code = df$adm2_code, key = paste0(prefix, "_", format(df$month, "%Y_%m")), v = df[[value]])
  m  <- sort(unique(df$month))
  list(wide   = tidyr::pivot_wider(d, names_from = "key", values_from = "v"),
       fields = lapply(m, function(x) list(field = paste0(prefix, "_", format(x, "%Y_%m")), label = format(x, "%b %Y"))))
}
# x/y/value data frame -> heat-point JSON for the browser + a GeoTIFF for GIS users
heat_export <- function(df, name, crs = 32630, max_pts = 25000) {
  df <- df[is.finite(df$value), c("x", "y", "value")]
  tryCatch(terra::writeRaster(terra::rast(df, type = "xyz", crs = paste0("EPSG:", crs)),
                              file.path(out_dir, paste0(name, ".tif")), overwrite = TRUE),
           error = function(e) message("GeoTIFF skipped for ", name, ": ", conditionMessage(e)))
  if (nrow(df) > max_pts) df <- df[round(seq(1, nrow(df), length.out = max_pts)), ]
  ll <- sf::st_coordinates(wgs(sf::st_as_sf(df, coords = c("x", "y"), crs = crs)))
  jsonlite::write_json(list(points = data.frame(lat = ll[, 2], lng = ll[, 1], value = df$value)),
                       file.path(out_dir, paste0(name, ".heat.json")), digits = 6)
}
im_df  <- function(im) { d <- as.data.frame(im); names(d)[3] <- "value"; d }          # spatstat image
pts_df <- function(s, col) { xy <- sf::st_coordinates(s); data.frame(x = xy[, 1], y = xy[, 2], value = s[[col]]) }

# The decision: Northern Region priority with its evidence, then age-standardised risk for every district
safe("measles decision", {
  dec <- nr_dec |> dplyr::transmute(adm2_code, priority, P_top3, rank_point, RR_mean, RR_lo, RR_hi, rate_u5, O,
                                    mcv1, mcv2, dropout, susceptible_n, p_alert, fc_cases, months_usable_24, months_lt80)
  d <- web_districts(districts |> dplyr::filter(region == focus_region)) |> dplyr::left_join(dec, by = "adm2_code") |>
    dplyr::mutate(recommendation = dplyr::if_else(priority <= 3, "Recommended top 3", "Other"))
  write_geojson(d, "measles_northern_decision")
  add_layer(id = "decision", title = "Northern Region: measles priority and evidence", kind = "choropleth",
    file = "measles_northern_decision.geojson", source = "decision_table_northern.csv",
    fields = list(
      list(field = "recommendation",   label = "Recommendation", type = "category",
           colors = list("Recommended top 3" = "#B2182B", "Other" = "#d9d9d9")),
      list(field = "P_top3",           label = "P(in the Northern top 3)", type = "numeric"),
      list(field = "RR_mean",          label = "Posterior relative risk (age-standardised)", type = "numeric"),
      list(field = "rate_u5",          label = "Under-5 rate per 100,000 person-years", type = "numeric"),
      list(field = "susceptible_n",    label = "Estimated susceptible children 0-14", type = "numeric"),
      list(field = "dropout",          label = "MCV1-MCV2 dropout 2024 (%)", type = "numeric"),
      list(field = "mcv1",             label = "MCV1 coverage 2024 (%)", type = "numeric"),
      list(field = "p_alert",          label = "P(alert), January 2025", type = "numeric"),
      list(field = "months_usable_24", label = "Usable reporting months (of 24)", type = "numeric"))) })

safe("measles risk", {
  a <- ann |> dplyr::select(adm2_code, O, E_age, SIR_age, RR_mean, RR_lo, RR_hi, P_RR_gt1, rate_u5)
  d <- web_districts(districts) |> dplyr::left_join(a, by = "adm2_code")
  write_geojson(d, "measles_risk_2024")
  add_layer(id = "risk", title = "Measles risk 2024, all regions (age-standardised)", kind = "choropleth",
    file = "measles_risk_2024.geojson",
    fields = list(
      list(field = "RR_mean",  label = "Posterior relative risk", type = "numeric"),
      list(field = "SIR_age",  label = "Crude SIR (age-standardised)", type = "numeric"),
      list(field = "P_RR_gt1", label = "P(relative risk > 1)", type = "numeric"),
      list(field = "rate_u5",  label = "Under-5 rate per 100,000 person-years", type = "numeric"),
      list(field = "O",        label = "Confirmed cases 2024, usable months", type = "numeric"))) })

jsonlite::write_json(list(layers = layers), file.path(out_dir, "layers.json"),
                     auto_unbox = TRUE, null = "null", na = "null")
cat("Exported", length(layers), "map layer(s) to", out_dir, "\n")
print(list.files(out_dir))
